# Experiment 1 (Toy Model): True vs. Data-Estimated vs. Trained-Model Lipschitz Constant

## Motivation

A model's **Lipschitz constant** `L` bounds how much its output can change
in response to a small change in its input:
`|f(x') - f(x)| <= L * ||x' - x||` for all `x, x'`. A smaller Lipschitz
constant means a smoother, more robust function — this is one of the
standard tools used to reason about a neural network's sensitivity to
input perturbations (e.g. adversarial examples). In practice we never have
the true function `f*`, only a trained model `f_hat` fit to a finite
sample of it, so any Lipschitz constant we compute is really an
*estimate*, and its quality depends on both how much data we had and
where that data was located.

This experiment builds a **toy regression problem where the true function
`f*` and its true Lipschitz constant `L*` are known essentially exactly**,
so we can directly measure how far off our estimates are — something
that's impossible to check on a real dataset like MNIST, where `L*` is
unknown. It is Experiment 1 of a larger project; later experiments (on
MNIST/Fashion-MNIST) will reuse the same estimator code (`estimators.py`,
eventually merged into a shared `lipschitz_diagnostics.py`) on real
trained classifiers, where ground truth is not available. This toy
experiment is what calibrates trust in those estimators.

## The three quantities being compared

1. **`L*`** — the *true* Lipschitz constant of the ground-truth function.
   - **Tier A** (a single `A * tanh(w^T x + b)` ridge): closed-form,
     `L* = A * ||w||`.
   - **Tier B** (a sum of 2-3 such ridges): no closed form, so it's
     estimated numerically — dense grid search over the domain, refined
     by gradient ascent (`torch.optim.LBFGS`) on `||grad f*(x)||` from
     many random restarts. This is treated as "true" for the rest of the
     experiment because it is exact to numerical precision.
2. **`L_hat_data`** — a *data-only* estimate: given only the sampled
   `(x_i, f*(x_i))` pairs (no model), take the steepest pairwise slope,
   `max_{i != j} |y_i - y_j| / ||x_i - x_j||`. This is the best you could
   do without fitting anything.
3. **`L_hat_model`** — estimates computed from a model `f_hat` *trained*
   on that sample, via three genuinely different sub-methods that are
   kept separate throughout this notebook (never averaged or conflated):
   - **pairwise** — the same steepest-slope formula as `L_hat_data`, but
     applied to the model's predictions instead of the raw data (so it
     can be evaluated anywhere, e.g. on points with no training data
     nearby).
   - **local-perturbation** — a finite-difference estimate: sample many
     points in a small ball around a query point `x0`, take the steepest
     slope from `x0` to any of them.
   - **gradient-norm** — the infinitesimal/exact-derivative estimate:
     `||grad f_hat(x0)||` via autograd. This is the local Lipschitz
     constant of the model's tangent plane at `x0`, not a finite
     difference.

## What the experiment is designed to reveal

Because `L*` is known exactly here, we can ask questions that are normally
unanswerable: Does `L_hat_model` converge to `L*` as training data grows?
Does model capacity matter? And — the main point of this experiment —
what happens to `L_hat_model` in a region where training data was
deliberately **sparse** (a "gap")? Does the trained model *flatten out*
there (underestimate `L*`), or does it *oscillate* and overshoot it? This
notebook builds that gap deliberately and measures the effect directly
(Step 6 below).

## How this notebook is organized

This is a **thin driver notebook** — every reusable function/class lives
in the sibling `.py` modules inside this same folder, and this notebook
only calls into them and displays the results:

| Module | Contents |
|---|---|
| `toy_functions.py` | Ground-truth `f*` (Tier A single ridge, Tier B multi-ridge sum), analytic gradients, and `L*` computation. |
| `data.py` | Sampling schemes — uniform, and "gap" sampling that deliberately undersamples a region. |
| `estimators.py` | The `L_hat_data` / `L_hat_model` estimator functions described above. |
| `models.py` | `TinyMLP` (the trained model `f_hat`) and `SingleTanhUnit` (a model that exactly matches the Tier A functional form, used only as a sanity check). |
| `plots.py` | All plotting functions used below. |
| `run_experiment.py` | The driver functions this notebook calls (`run_tier_a_sanity`, `run_main_experiment`, `run_sweeps`, `run_2d_extension`). |
| `tests/` | Unit tests: the analytic gradient is checked against autograd, and the estimators are checked against the closed-form Tier A answer. |

Every design choice (float64 throughout, the domain/norm conventions, why
`scipy` isn't a dependency, etc.) is documented as docstrings/comments in
those modules and in `README.md` in this folder.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2

%matplotlib inline
import matplotlib.pyplot as plt

from toy_lipschitz import run_experiment

## Step 5: Tier A sanity check — does the whole pipeline agree with itself?

Before trusting any Tier B result (which has no closed-form check), we
verify the entire pipeline — sampling, training, and all three
`L_hat_model` sub-methods — against a case where the answer is known
exactly: a single ridge `f*(x) = A * tanh(w^T x + b)`, whose true
Lipschitz constant is `L* = A * ||w||`.

We sample 500 points, train a `SingleTanhUnit` (a model with *exactly* the
same functional form as `f*`, so it can fit it almost perfectly), and then
compute `L*`, `L_hat_data`, and all three `L_hat_model` sub-methods. If any
of them disagree with `L*` by more than 10%, `run_tier_a_sanity` raises an
`AssertionError` and this notebook stops here — everything below assumes
this cell passed, with all reported relative errors well under that
tolerance (in practice, under ~1%).

In [2]:
tier_a_results = run_experiment.run_tier_a_sanity()

=== Tier A sanity run ===
  L_star: 6.0
  L_hat_data: 5.94328560167768
  L_hat_model_pairwise: 5.943283551096246
  L_hat_model_local: 5.999997726601632
  L_hat_model_grad: 5.9999978825478655
  final_train_mse: 2.5702824534229708e-15
  L_hat_data rel_err=0.0095 (tol=0.1)
  L_hat_model_pairwise rel_err=0.0095 (tol=0.1)
  L_hat_model_local rel_err=0.0000 (tol=0.1)
  L_hat_model_grad rel_err=0.0000 (tol=0.1)


### Tier A gap demo — the flattening effect in the simplest possible case

Step 6 below shows the gap-vs-uniform undershoot effect on the harder Tier
B (multi-ridge) function. Before that, here's the same comparison on the
simplest possible ground truth: a single ridge `f*(x) = A * tanh(w^T x + b)`,
whose Lipschitz constant `L* = A * ||w||` is exact and closed-form (no grid
search needed).

This uses `TinyMLP`, not `SingleTanhUnit` from the Step 5 sanity check —
`SingleTanhUnit` matches `f*`'s functional form exactly and would recover
`L*` from almost any data, gap or not, so it can't demonstrate undershoot.
`TinyMLP` has to learn the shape from data, so a sampling gap placed
exactly at the true argmax `x*` can genuinely starve it of signal there.
`w=(15.0,)` makes this ridge much steeper than the Step 5 example, so the
undershoot is easy to see.

In [3]:
tier_a_gap_demo_results = run_experiment.run_tier_a_gap_demo(w=(15.0,), A=1.0, gap_radius=0.3, gap_fraction=0.01)


=== Tier A gap demo: L*=15.0000, x*=[-0.03333333333333333] ===

--- gap dataset ---
  L_hat_data: 4.791501247302037
  L_hat_model_pairwise_train: 4.758303107218451
  L_hat_model_pairwise_grid: 8.179338181965486
  final_train_mse: 8.220782583349383e-07
  max_local_lipschitz_in_gap: 8.181372846402056
  max_local_lipschitz_outside_gap: 3.2228300294239087

--- uniform dataset ---
  L_hat_data: 14.801754631437074
  L_hat_model_pairwise_train: 14.8288820741747
  L_hat_model_pairwise_grid: 14.960501026403975
  final_train_mse: 6.421878304833502e-08
  max_local_lipschitz_in_gap: 15.028927344038776
  max_local_lipschitz_outside_gap: 0.4374399417532427


## Step 6: Main experiment — gap vs. uniform sampling (Tier B, d=1)

Now the real question. `run_main_experiment` does the following:

1. Builds a **Tier B ground truth**: a sum of three `tanh` ridges of
   different steepness (`||w||` = 8, 2, and 5) placed at different
   offsets `b`, so they don't overlap. There is no closed form for this
   function's Lipschitz constant, so `L*` and the point `x*` where it is
   attained are found via the grid-search + gradient-ascent procedure in
   `tier_b_true_L`.
2. Builds **two datasets of the same size `N`**: a `uniform` dataset
   (ordinary i.i.d. sampling over the whole domain) and a `gap` dataset,
   which is identical except that the region around `x*` — precisely
   where the function is steepest — is deliberately undersampled (only a
   small `gap_fraction` of points fall there).
3. Trains a `TinyMLP` on each dataset independently.
4. For each trained model, evaluates `L_hat_data` (from the raw training
   sample), `L_hat_model` pairwise (on the training points and on a dense
   grid spanning the whole domain, including inside the gap), and the
   local-perturbation and gradient-norm estimates *at every point on that
   grid* — not just their max, so we can see the whole shape.

The resulting plot overlays `f*(x)`, the trained `f_hat(x)`, and the
pointwise local-Lipschitz (finite-difference) curve for both datasets side
by side, with the training points shown as a rug plot along the x-axis so
you can see exactly where each model did and didn't get to see data.
**What to look for:** compare the height of the red local-Lipschitz curve
near `x*` between the two panels — if the gap dataset's peak is visibly
lower than the uniform dataset's, that is the model *flattening out*
(underestimating `L*`) precisely where its training data was sparse.

In [4]:
main_results = run_experiment.run_main_experiment()


=== Tier B main experiment: L*=8.4628, x*=[-0.3706231364023103] ===

--- gap dataset ---
  L_hat_data: 8.17499629931324
  L_hat_model_pairwise_train: 8.090717673114709
  L_hat_model_pairwise_grid: 8.199876753394625
  final_train_mse: 7.292470633003489e-06
  max_local_lipschitz_in_gap: 8.205083489792763
  max_local_lipschitz_outside_gap: 3.480261681098084

--- uniform dataset ---
  L_hat_data: 8.327131329747235
  L_hat_model_pairwise_train: 8.60478434242492
  L_hat_model_pairwise_grid: 8.718726954399061
  final_train_mse: 2.7755613717803226e-06
  max_local_lipschitz_in_gap: 8.737787823364094
  max_local_lipschitz_outside_gap: 4.558074728840353


### Step 6.7: quantifying the gap effect

`report` below lists, per dataset, the max local-Lipschitz estimate inside
the gap region vs. outside it — a lower in-gap max for the gap dataset
relative to the uniform dataset is the numeric confirmation of the
flattening effect described above (rather than just eyeballing the plot).

In [5]:
for dataset_key, r in main_results["report"].items():
    print(dataset_key, r,'\n')

gap {'L_hat_data': 8.17499629931324, 'L_hat_model_pairwise_train': 8.090717673114709, 'L_hat_model_pairwise_grid': 8.199876753394625, 'final_train_mse': 7.292470633003489e-06, 'max_local_lipschitz_in_gap': 8.205083489792763, 'max_local_lipschitz_outside_gap': 3.480261681098084} 

uniform {'L_hat_data': 8.327131329747235, 'L_hat_model_pairwise_train': 8.60478434242492, 'L_hat_model_pairwise_grid': 8.718726954399061, 'final_train_mse': 2.7755613717803226e-06, 'max_local_lipschitz_in_gap': 8.737787823364094, 'max_local_lipschitz_outside_gap': 4.558074728840353} 



## Step 7: Sweeps — does more data or more capacity fix the gap?

Two sweeps, each repeated for both the `uniform` and `gap` datasets:

- **N-sweep**: fix model capacity, vary training-set size
  `N ∈ {50, 100, 200, 500, 1000, 2000, 5000}`.
- **Capacity-sweep**: fix `N`, vary the `TinyMLP` hidden width
  `∈ {4, 8, 16, 32, 64, 128}`.

In both, `L_hat_model` is evaluated on a **fixed, large, held-out grid** —
not on the training points — specifically so the numbers reflect how well
the model *generalizes* to steep regions it may not have trained on, not
how well it merely memorized its own training set.

Each plot shows `L*` as a horizontal reference line alongside the
`L_hat_data` / `L_hat_model` curves. **What to look for:** on the
`uniform` dataset, both curves should climb toward `L*` as N or width
increases. On the `gap` dataset, watch whether `L_hat_model` climbs toward
`L*` as capacity increases at fixed `N` — if it stays stuck below `L*`
across the whole width range, that's evidence that the information
genuinely isn't there to recover (a coverage problem, not a capacity
problem).

In [6]:
sweep_results = run_experiment.run_sweeps()

=== Step 7 sweeps: ground truth L* = 8.4628 (x* = [-0.3706231364023103]) ===

=== N-sweep (uniform) ===
  [uniform] N=   50  L_hat_data=8.270  L_hat_model=8.583
  [uniform] N=  100  L_hat_data=8.270  L_hat_model=8.495
  [uniform] N=  200  L_hat_data=8.283  L_hat_model=8.482
  [uniform] N=  500  L_hat_data=8.327  L_hat_model=8.150
  [uniform] N= 1000  L_hat_data=8.362  L_hat_model=8.595
  [uniform] N= 2000  L_hat_data=8.455  L_hat_model=8.654
  [uniform] N= 5000  L_hat_data=8.463  L_hat_model=8.699

=== capacity-sweep (uniform) ===
  [uniform] width=   4  L_hat_data=8.327  L_hat_model=6.700
  [uniform] width=   8  L_hat_data=8.327  L_hat_model=6.712
  [uniform] width=  16  L_hat_data=8.327  L_hat_model=7.772
  [uniform] width=  32  L_hat_data=8.327  L_hat_model=7.503
  [uniform] width=  64  L_hat_data=8.327  L_hat_model=8.150
  [uniform] width= 128  L_hat_data=8.327  L_hat_model=8.598

=== N-sweep (gap) ===
  [gap] N=   50  L_hat_data=3.661  L_hat_model=6.907
  [gap] N=  100  L_hat_data

## Step 8: 2D extension — the version that generalizes to MNIST/Fashion-MNIST

Everything above is 1D, which is easy to plot as a curve but not
representative of real high-dimensional inputs. This step repeats the
Tier B setup in `d=2`, with three ridges pointing in different directions
(steep along `x1`, shallow along `x2`, moderate diagonally), trains a
`TinyMLP` on a 2D dataset with the same kind of deliberate sampling gap,
and produces three heatmaps over `[-5,5]^2`:

1. the **true** `||grad f*(x)||` (ground truth, from the analytic
   gradient),
2. the **model's** `||grad f_hat(x)||` (autograd, the gradient-norm
   estimator),
3. the **finite-difference local Lipschitz estimate** (the
   local-perturbation estimator),

with the training points scattered on top of each so gap regions and
"hot spots" in the estimates can be visually compared directly. This is
the closest analogue to what Experiments 2-4 will do on real image
classifiers, where the input is high-dimensional and there's no way to
draw a single 1D curve — a heatmap (or its higher-dimensional analogue) is
the only way to see where a model's local sensitivity is highest.

In [7]:
results_2d = run_experiment.run_2d_extension()


=== 2D extension: L*=11.4250, x*=[-0.3750937556107243, 0.3764795582689485] ===
  final train MSE: 0.000004


## Summary and status

Running this notebook end-to-end (with the fixed seed used throughout)
reproduces the two central findings of this experiment:

- **The gap causes flattening, not oscillation.** In Step 6, the trained
  model's local-Lipschitz estimate near `x*` is visibly *lower* for the
  gap-sampled dataset than for the uniformly-sampled one — the model
  underestimates the true steepness where it lacked data, rather than
  overshooting it.
- **Capacity does not fix a coverage gap.** In Step 7's capacity-sweep on
  the gap dataset, `L_hat_model` stays below both `L_hat_data` and `L*`
  across the entire range of hidden widths tested (4 to 128) — more
  parameters alone cannot recover information that was never in the
  training sample.

This is **Experiment 1** of a larger project studying how well a
Lipschitz constant can be estimated from a trained model, motivated by the
fact that a model's Lipschitz constant bounds its sensitivity to small
input perturbations (relevant to adversarial robustness). `estimators.py`
in this folder is written generally enough (arbitrary function, arbitrary
input dimension) that Experiments 2-4 — which will run these same
estimators on models trained on MNIST/Fashion-MNIST, where the true `L*`
is *not* known — can import it directly rather than reimplementing it.
The toy problem here is what calibrates how much to trust those
estimators when ground truth isn't available.

See `README.md` in this folder for a plain file-by-file reference of the
codebase.